In [3]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

##### ANALISIS DE EXPERIMENTOS

##### 7-1 Introduccion

AB Testing:

El AB Testing (o prueba A/B) es una técnica estadística muy utilizada en marketing, desarrollo web, ciencia de datos y 
otras áreas para comparar dos versiones de algo (como una página web, un anuncio o una funcionalidad) y determinar cuál funciona mejor. 

Conceptos básicos del AB Testing

Objetivo:

Comparar dos grupos (A y B) para ver si hay diferencias significativas en una métrica clave (ej.: tasa de conversión, clics, ventas).

Ejemplo:

Versión A (actual): Botón verde en un sitio web.

Versión B (nueva): Botón rojo.

Métrica a comparar: % de usuarios que hacen clic.

Hipótesis:

Hipótesis nula (H₀): No hay diferencia entre A y B.

Hipótesis alternativa (H₁): Hay una diferencia significativa.

Métricas comunes:

Tasa de conversión, retención, tiempo en página, ingresos por usuario, etc.

##### 7-2 Por que si y por que no usar SQL

SQL se usa para extraer los datos y manipularlos, para análisis estadístico se requiere software especializado como R o Python.

##### 7-3 Correlacion vs Causalidad

El principio "correlación no implica causalidad" es uno de los conceptos más importantes en estadística y ciencia de datos. 
Significa que aunque dos variables parezcan estar relacionadas (es decir, cambian juntas), no necesariamente una causa la otra. 

3 Razones por las que correlación ≠ causalidad

1.-Variables confusoras (tercer factor):
	
 Una tercera variable influye en ambas.
	
 Ejemplo:
	
 Correlación entre ventas de sombrillas y tormentas.
	
 La variable confusora es la lluvia (aumenta ambas).
 
 2.-Causalidad inversa:
	
 La dirección de la relación es al revés de lo que se asume.
	
 Ejemplo:
	
 Correlación entre "más policías" y "más crímenes".
	
 En realidad, las ciudades con más crimen contratan más policías.

3.-Pura coincidencia (correlación espuria):
	
 La relación es aleatoria y no tiene lógica.

Ejemplo:

Correlación entre el número de piratas y el calentamiento global.

Para probar que A causa B, se necesitan:

-Experimentos controlados:

Como los A/B tests, donde se aísla la variable de interés (ej.: cambiar solo el color de un botón).

-Métodos estadísticos avanzados:

Modelos de regresión con variables instrumentales.

Análisis de series temporales (para descartar tendencias).

-Teoría sólida:

Un mecanismo lógico que explique por qué A debería causar B.

##### 7-4 Experimentos Binarios

Experimento binario: Buscamos ver si se realizó o no una acción de interés. Ejemplo: Los clientes dan click o no dan click en una página web.

Experimento contínuo: Estudiar una variable cuantitativa. Por ejemplo, cuanto dinero gastó un cliente en mi sitio web.

Como herramientas estadísticas se usará la prueba ji-cuadrada y la prueba t de student

In [4]:
# Trabajaremos con datos de una página web

query = r"""
select
     * 
from
     experimentos.paginaweb;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,user_id,variant,views,clicks,purchased_items,experiment_date,lastpurch_date,country
0,1,control,12,0,4,2020-01-01,2020-01-10,Argentina
1,2,control,3,1,3,2020-01-01,2020-01-31,Chile
2,3,control,12,0,2,2020-01-01,2020-01-18,México
3,4,control,2,0,3,2020-01-01,2020-01-18,Perú
4,5,control,4,0,4,2020-01-01,2020-01-26,México
5,6,control,1,1,1,2020-01-01,2020-01-16,Guatemala
6,7,control,27,1,3,2020-01-01,2020-01-26,Argentina
7,8,control,8,0,1,2020-01-01,2020-01-01,Perú
8,9,control,2,0,0,2020-01-01,2020-01-30,México
9,10,control,4,0,4,2020-01-01,2020-01-31,Argentina


In [6]:
# Agrupar por variable variant y obtener cuantos registros tienen click y cuantos no */

query = r"""
select
    variant,
    count(case when clicks=1 then 1 end) as si_click,
    count(case when clicks=0 then 1 end) as no_click
from
    experimentos.paginaweb
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,variant,si_click,no_click
0,control,32628,32372
1,test,32593,32407


In [8]:
# Agrupando por la variable variant obtener el total de usuario, el total de clicks y el porcentaje de clicks */

query = r"""
select
    variant,
    count(*) as intentos_totales,
    count(case when clicks=1 then 1 end) as num_exitos,
    count(case when clicks=1 then 1 end)/count(*) as porcentaje_exitos
from
    experimentos.paginaweb
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,variant,intentos_totales,num_exitos,porcentaje_exitos
0,control,65000,32628,0.5020
1,test,65000,32593,0.5014


La tasa de clics se calcula como:
$$
\hat p=\frac{\text{número de clics}}{\text{intentos totales}}
$$

Para el grupo de control:
$$
\hat p_C=\frac{32628}{65000}=0.501969
$$

Y para el grupo de prueba:
$$
\hat p_T=\frac{32593}{65000}=0.501431
$$

La diferencia observada es:
$$
\begin{aligned}
\hat p_T-\hat p_C
&=0.501431-0.501969\\
&=-0.000538
\end{aligned}
$$

Esto equivale a una diferencia de aproximadamente −0.0538 puntos porcentuales. La página nueva presenta una tasa de clics ligeramente menor, pero todavía debemos comprobar si esa diferencia puede explicarse por variabilidad aleatoria.

Como queremos saber si el número de clics es distinto, debemos usar una prueba bilateral.

Definimos:
- \(p_C\): probabilidad de hacer clic con la página original.
- \(p_T\): probabilidad de hacer clic con la página nueva.
Las hipótesis son:
$$
H_0:p_C=p_T
$$

$$
H_1:p_C\ne p_T
$$

La hipótesis nula \(H_0\) afirma que ambas páginas tienen la misma probabilidad de generar clics.

La hipótesis alternativa \(H_1\) afirma que las probabilidades son diferentes, ya sea porque la nueva página mejora o empeora la tasa de clics.

Antes de calcular, elegimos un nivel de significancia. Usaremos el habitual:
$$
\alpha=0.05
$$

En este caso usamos la prueba Z para dos proporciones independientes.

¿Por qué?
- La variable de respuesta es binaria: hubo clic o no hubo clic.
- Tenemos dos grupos: control y test.
- Queremos comparar dos proporciones.
- Los tamaños de muestra son grandes: 65,000 observaciones por grupo.
La prueba compara la diferencia observada con la diferencia que esperaríamos encontrar debido al azar si ambas páginas tuvieran la misma tasa de clics.
Calcular el estadístico Z

Bajo la hipótesis nula, calculamos primero la proporción combinada:
$$
\hat p=\frac{x_C+x_T}{n_C+n_T}
$$

Sustituyendo los valores:
$$
\hat p=\frac{32628+32593}{65000+65000}
=\frac{65221}{130000}
\approx 0.5017
$$

El error estándar bajo la hipótesis nula es:
$$
SE_0=\sqrt{\hat p(1-\hat p)
\left(\frac{1}{n_C}+\frac{1}{n_T}\right)}
$$

Y el estadístico de prueba es:
$$
Z=\frac{\hat p_T-\hat p_C}{SE_0}
$$

Al sustituir los datos, obtenemos:
$$
\boxed{Z\approx -0.1941}
$$

El signo negativo indica que la tasa de clics del grupo test fue menor que la del control. Sin embargo, el valor está muy cerca de cero, lo que indica que la diferencia observada es pequeña en relación con la variabilidad esperada.
Ahora calculamos el valor p:

Como la prueba es bilateral, calculamos:
$$
p\text{-valor}=2P(Z\geq |z_{\text{obs}}|)
$$

Para $$(Z=-0.1941)$$, el resultado es:
$$
\boxed{p\text{-valor}\approx 0.8461}
$$

Comparemos este valor con nuestro nivel de significancia:
$$
0.8461>0.05
$$

Por lo tanto, no rechazamos la hipótesis nula.

Esto significa que no encontramos evidencia estadística suficiente para concluir que la tasa de clics sea diferente entre ambas páginas.